# Importing Required Libraries

This code imports essential Python libraries for data handling, visualization, machine learning metrics, and deep learning. It sets up the environment for building and evaluating models.

In [ ]:
import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize
from scipy.stats import ttest_rel

import tensorflow as tf
from tensorflow.keras import layers, models

from tensorflow.keras.applications import ResNet50, VGG16, EfficientNetB3
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as eff_preprocess

from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Dataset Paths and Configuration

Defines image size, batch size, and directory paths for training, validation, and test datasets. This prepares the dataset structure for model training.

In [ ]:
IMG_SIZE = (300, 300)
BATCH_SIZE = 32

base_dir = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

train_dir = base_dir + "/train"
val_dir   = base_dir + "/val"
test_dir  = base_dir + "/test"

# --- EfficientNet preprocessing used as standard (BEST PERFORMANCE) ---
train_datagen = ImageDataGenerator(
    preprocessing_function=eff_preprocess,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=eff_preprocess
)

train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_data = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_data = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

# Class weights (IMPORTANT for imbalance)
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_data.classes),
    y=train_data.classes
)

class_weights = dict(enumerate(class_weights))
print("Class Weights:", class_weights)

# Model Building Function

Creates a transfer learning model using a pretrained base (e.g., EfficientNet, ResNet). Most layers are frozen, while top layers are fine-tuned for better performance on the target dataset.

In [ ]:
def build_model(base_model, fine_tune_layers=80):

    # Freeze most layers
    for layer in base_model.layers[:-fine_tune_layers]:
        layer.trainable = False

    # Fine-tune top layers
    for layer in base_model.layers[-fine_tune_layers:]:
        layer.trainable = True

    x = base_model.output
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    output = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

# Model Training Function

Handles the training process with callbacks like early stopping to prevent overfitting. It also tracks training time and performance.

In [ ]:
def train_model(model, name):

    start_time = time.time()

    callbacks = [
        tf.keras.callbacks.EarlyStopping(
            patience=4,
            restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            patience=2,
            factor=0.3
        )
    ]

    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=20,
        class_weight=class_weights,
        callbacks=callbacks,
        verbose=1
    )

    train_time = time.time() - start_time
    return history, train_time

# Model Evaluation Function

Evaluates the trained model on test data, generates predictions, and computes performance metrics such as accuracy and inference time.

In [ ]:
def evaluate_model(model, name):

    start_time = time.time()

    preds = model.predict(test_data)
    preds_binary = (preds > 0.5).astype(int).flatten()

    test_time = time.time() - start_time

    y_true = test_data.classes

    acc = accuracy_score(y_true, preds_binary)
    prec = precision_score(y_true, preds_binary)
    rec = recall_score(y_true, preds_binary)
    f1 = f1_score(y_true, preds_binary)

    cm = confusion_matrix(y_true, preds_binary)

    class_acc = cm.diagonal() / cm.sum(axis=1)

    auc = roc_auc_score(y_true, preds)

    fpr, tpr, _ = roc_curve(y_true, preds)

    return {
        "accuracy": acc,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "confusion_matrix": cm,
        "class_accuracy": class_acc,
        "auc": auc,
        "fpr": fpr,
        "tpr": tpr,
        "test_time": test_time
    }

# Pretrained Models Dictionary

Defines multiple pretrained models (e.g., EfficientNetB3, ResNet50) using the build function. This allows easy comparison between different architectures.

In [ ]:
models_dict = {
    
    "EfficientNetB3": build_model(
        EfficientNetB3(weights='imagenet', include_top=False, input_shape=(300,300,3)),
        fine_tune_layers=50
    ),

      "ResNet50": build_model(
        ResNet50(weights='imagenet', include_top=False, input_shape=(300,300,3)),
        fine_tune_layers=50
    ),

    "VGG16": build_model(
        VGG16(weights='imagenet', include_top=False, input_shape=(300,300,3)),
        fine_tune_layers=50
    )
}

results = {}
train_times = {}

for name, model in models_dict.items():
    print(f"\nTraining {name}...")
    _, train_time = train_model(model, name)
    train_times[name] = train_time

    print(f"Evaluating {name}...")
    results[name] = evaluate_model(model, name)


# ROC Curve Plotting

Plots ROC curves for all models to visualize their performance. It compares models using AUC (Area Under Curve) scores.

In [ ]:
plt.figure(figsize=(8,6))

for name in results:
    plt.plot(
        results[name]["fpr"],
        results[name]["tpr"],
        label=f"{name} (AUC={results[name]['auc']:.3f})",
        linewidth=2
    )

plt.plot([0,1],[0,1],'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid()
plt.show()


# Performance Summary Table

Creates a summary of evaluation metrics (accuracy, class-wise accuracy, etc.) for all models in a structured format.

In [ ]:
summary = []

for name in results:
    summary.append({
        "Model": name,
        "Accuracy": results[name]["accuracy"],
        "Class Acc (Normal)": results[name]["class_accuracy"][0],
        "Class Acc (Pneumonia)": results[name]["class_accuracy"][1],
        "Precision": results[name]["precision"],
        "Recall": results[name]["recall"],
        "F1-score": results[name]["f1"],
        "AUC": results[name]["auc"],
        "Train Time (s)": train_times[name],
        "Test Time (s)": results[name]["test_time"]
    })

df_summary = pd.DataFrame(summary)
print(df_summary)

# Statistical Comparison of Models

Performs statistical tests (e.g., paired t-test) to compare predictions from different models and check if performance differences are significant.

In [ ]:
predictions = {}

for name, model in models_dict.items():
    preds = model.predict(test_data).flatten()
    predictions[name] = preds

# Example: ResNet vs VGG
t_stat, p_value = ttest_rel(predictions["ResNet50"], predictions["VGG16"])

print("T-test ResNet vs VGG:")
print("t-stat:", t_stat)
print("p-value:", p_value)

# Confusion Matrix Visualization

Defines a function to plot confusion matrices using heatmaps, helping visualize classification performance (true vs predicted labels).

In [ ]:
def plot_confusion_matrix(cm, model_name):
    plt.figure(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Normal", "Pneumonia"],
                yticklabels=["Normal", "Pneumonia"])
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(f"Confusion Matrix - {model_name}")
    plt.show()

# Plot for all models
for name in results:
    plot_confusion_matrix(results[name]["confusion_matrix"], name)